In [0]:
%pip install geopandas shapely

In [0]:
%restart_python

In [0]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import box

BRONZE_BASE_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "bronze/fire_hazard"
)

SILVER_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/fire_hazard"
)

MIN_LONGITUDE = -118.95
MIN_LATITUDE = 33.95
MAX_LONGITUDE = -118.45
MAX_LATITUDE = 34.25

print("Libraries loaded.")

In [0]:
folders = dbutils.fs.ls(BRONZE_BASE_PATH)

load_folders = sorted(
    [x.path for x in folders if "load_timestamp=" in x.path]
)

latest_path = load_folders[-1]

print("Latest Bronze path:")
print(latest_path)

In [0]:
raw_df = spark.read.format("delta").load(latest_path)

raw_json = raw_df.select("raw_json").first()["raw_json"]

hazard_json = json.loads(raw_json)

print("Raw features:", len(hazard_json["features"]))

In [0]:
hazard_gdf = gpd.GeoDataFrame.from_features(
    hazard_json["features"],
    crs="EPSG:4326"
)

print("Rows:", len(hazard_gdf))
print("Columns:")
print(hazard_gdf.columns.tolist())

In [0]:
study_boundary = gpd.GeoDataFrame(
    {"study_area": ["Western Los Angeles County"]},
    geometry=[
        box(
            MIN_LONGITUDE,
            MIN_LATITUDE,
            MAX_LONGITUDE,
            MAX_LATITUDE
        )
    ],
    crs="EPSG:4326"
)

# Keep only hazard polygons that intersect our Western LA study area
hazard_west = hazard_gdf[
    hazard_gdf.geometry.intersects(study_boundary.geometry.iloc[0])
].copy()

# Repair any invalid geometries
hazard_west["geometry"] = hazard_west.geometry.make_valid()

# Clip polygons to the study-area boundary
hazard_west = gpd.clip(
    hazard_west,
    study_boundary
)

# Remove empty/null geometries
hazard_west = hazard_west[
    hazard_west.geometry.notna() &
    ~hazard_west.geometry.is_empty
].copy()

print("Raw LA County hazard features:", len(hazard_gdf))
print("Western LA hazard features:", len(hazard_west))

In [0]:
print("\nHazard classes:")
print(hazard_west["HAZ_CLASS"].value_counts(dropna=False))

print("\nResponsibility areas:")
print(hazard_west["SRA"].value_counts(dropna=False))

print("\nGeometry valid:")
print(hazard_west.geometry.is_valid.value_counts())

In [0]:
hazard_score_map = {
    "NonWildland": 0.0,
    "Moderate": 1.0,
    "High": 2.0,
    "Very High": 3.0
}

hazard_west["hazard_score"] = (
    hazard_west["HAZ_CLASS"]
    .map(hazard_score_map)
)

print(
    hazard_west[
        ["OBJECTID", "SRA", "HAZ_CLASS", "hazard_score"]
    ]
    .sort_values("hazard_score", ascending=False)
)

print("\nHazard score range:")
print(
    hazard_west["hazard_score"].min(),
    "to",
    hazard_west["hazard_score"].max()
)

In [0]:
silver_hazard = hazard_west[
    [
        "OBJECTID",
        "SRA",
        "HAZ_CLASS",
        "HAZ_CODE",
        "geometry",
        "hazard_score"
    ]
].copy()

silver_hazard = silver_hazard.rename(
    columns={
        "OBJECTID": "object_id",
        "SRA": "responsibility_area",
        "HAZ_CLASS": "hazard_class",
        "HAZ_CODE": "hazard_code"
    }
)

silver_hazard["geometry_wkt"] = silver_hazard.geometry.apply(
    lambda geom: geom.wkt if geom is not None else None
)

silver_hazard = silver_hazard.drop(columns=["geometry"])

print("Silver hazard rows:", len(silver_hazard))
print(silver_hazard.head())

In [0]:
silver_spark_df = spark.createDataFrame(silver_hazard)

silver_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH)

print("Silver hazard dataset written successfully")
print("Rows written:", silver_spark_df.count())
print("Path:", SILVER_PATH)

In [0]:
check_silver = spark.read.format("delta").load(SILVER_PATH)

print("Total rows:", check_silver.count())
print(
    "Unique object IDs:",
    check_silver.select("object_id").distinct().count()
)